In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.special import softmax
from sklearn.metrics import accuracy_score

In [2]:
ROOT_DIR = Path.cwd().parent

OOF_DIR = ROOT_DIR / "oof"

In [3]:
conv_logits_oof = np.load(OOF_DIR / "convnext_oof_logits.npy")
swin_logits_oof = np.load(OOF_DIR / "swin_oof_logits.npy")
siglip_logits_oof = np.load(OOF_DIR / "siglip_oof_logits.npy")

In [4]:
p_conv_oof = softmax(conv_logits_oof, axis=1)
p_swin_oof = softmax(swin_logits_oof, axis=1)
p_siglip_oof = softmax(siglip_logits_oof, axis=1)

In [5]:
train_folds = pd.read_csv("train_folds.csv")

y_true = train_folds["label"].to_numpy()

In [6]:
pred_conv = p_conv_oof.argmax(axis=1)
pred_swin = p_swin_oof.argmax(axis=1)
pred_siglip = p_siglip_oof.argmax(axis=1)

In [7]:
FINAL_WEIGHTS = np.array([0.45,0.43,0.12])

p_ensemble = (
      FINAL_WEIGHTS[0]*p_conv_oof
    + FINAL_WEIGHTS[1]*p_swin_oof
    + FINAL_WEIGHTS[2]*p_siglip_oof
)

pred_ensemble = p_ensemble.argmax(axis=1)

In [8]:
print(f"ConvNeXt : {accuracy_score(y_true,pred_conv):.6f}")
print(f"Swin      : {accuracy_score(y_true,pred_swin):.6f}")
print(f"SigLIP    : {accuracy_score(y_true,pred_siglip):.6f}")
print(f"Ensemble  : {accuracy_score(y_true,pred_ensemble):.6f}")

ConvNeXt : 0.973645
Swin      : 0.973404
SigLIP    : 0.957993
Ensemble  : 0.981089


In [9]:
agreement=[]

for a,b,c in zip(pred_conv,pred_swin,pred_siglip):

    if a==b==c:

        agreement.append("3 Models Agree")

    elif len({a,b,c})==2:

        agreement.append("2 Models Agree")

    else:

        agreement.append("All Different")

In [10]:
conv_conf = p_conv_oof.max(axis=1)

swin_conf = p_swin_oof.max(axis=1)

siglip_conf = p_siglip_oof.max(axis=1)

ensemble_conf = p_ensemble.max(axis=1)

In [11]:
conv_correct = pred_conv==y_true

swin_correct = pred_swin==y_true

siglip_correct = pred_siglip==y_true

ensemble_correct = pred_ensemble==y_true

In [12]:
result_df = pd.DataFrame({

    "true":y_true,

    "conv_pred":pred_conv,
    "swin_pred":pred_swin,
    "siglip_pred":pred_siglip,
    "ensemble_pred":pred_ensemble,

    "conv_correct":conv_correct,
    "swin_correct":swin_correct,
    "siglip_correct":siglip_correct,
    "ensemble_correct":ensemble_correct,

    "conv_conf":conv_conf,
    "swin_conf":swin_conf,
    "siglip_conf":siglip_conf,
    "ensemble_conf":ensemble_conf,

    "agreement":agreement

})

In [13]:
labels=[
    "Recyclable",
    "Electronic",
    "Organic"
]

for i,label in enumerate(labels):

    result_df[f"conv_{label}"]=p_conv_oof[:,i]

    result_df[f"swin_{label}"]=p_swin_oof[:,i]

    result_df[f"siglip_{label}"]=p_siglip_oof[:,i]

    result_df[f"ensemble_{label}"]=p_ensemble[:,i]

In [14]:
result_df.to_csv(
    "model_predictions.csv",
    index=False
)

print(result_df.head())

print()

print("Saved model_predictions.csv")

   true  conv_pred  swin_pred  siglip_pred  ensemble_pred  conv_correct  \
0     0          0          0            0              0          True   
1     0          0          0            0              0          True   
2     0          0          0            0              0          True   
3     0          0          0            0              0          True   
4     0          0          0            0              0          True   

   swin_correct  siglip_correct  ensemble_correct  conv_conf  ...  \
0          True            True              True   0.904741  ...   
1          True            True              True   0.913278  ...   
2          True            True              True   0.914421  ...   
3          True            True              True   0.949260  ...   
4          True            True              True   0.949947  ...   

   siglip_Recyclable  ensemble_Recyclable  conv_Electronic swin_Electronic  \
0           0.999995             0.936263         0.0768